# Module 3 — Solutions to the Exercises
### Attention, Transformers & the Particle Transformer (ParT) · TIFR ML School 2026

Full worked solutions to the eight exercises at the end of
`Module3_Attention_Transformers_ParT.ipynb`. **Self-contained**: the *Setup* re-defines scaled-dot-product /
multi-head attention, the encoder block, attention pooling, ParT's pairwise bias, and loads JetClass as
masked sets — then trains a **base plain Transformer** and a **base ParT** that the exercises reuse.

| # | Exercise | Idea it drives home |
|---|----------|---------------------|
| 1 | **Ablate the bias** | which of ParT's 4 pairwise features carries the advantage |
| 2 | **Depth / heads / width** | where capacity stops paying off for jets |
| 3 | **Class-attn vs mean-pool** | how much learnable pooling actually buys |
| 4 | **ISAB trade-off** | the AUC you pay for `O(N²)→O(Nm)` linear attention |
| 5 | **Attention → graph** | does attention rediscover ParticleNet's k-NN neighbours? |
| 6 | **Three-way race** | PFN vs ParticleNet vs ParT on one ROC, same jets |
| 7 | **Boost augmentation** | inputs aren't boost-invariant; ParT's bias already is |
| 8 | **Faithful attribution** | gradient-weighted rollout (Chefer et al.) vs plain rollout |

> **Design choices that keep this fast & clean:** the `ParticleTransformer` is **configurable**
> (`use_pair_bias`, `pair_feat_idx`, `pool`), so exercises 1 & 3 are just constructor flags; a single base
> plain/ParT pair is trained once and reused for interpretability (5, 8) and the race (6).
>
> **Speed knobs:** `N_PER_CLS`, `DIM`, `DEPTH`, `EPOCHS`. Defaults are deliberately small so all ~20 trainings
> finish in a few minutes on a laptop; raise them to approach the module's headline AUC (ParT ≈ 0.99).

## Setup — attention machinery, ParT, data, base models

In [ ]:
import os, math, time, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

torch.manual_seed(0); np.random.seed(0)
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("torch", torch.__version__, "| device", device)

In [ ]:
# --- attention primitives (from Module 3) -----------------------------------------------
def scaled_dot_product_attention(q, k, v, key_mask=None, attn_bias=None):
    d = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d)
    if attn_bias is not None: scores = scores + attn_bias
    if key_mask is not None:  scores = scores.masked_fill(~key_mask.unsqueeze(-2).bool(), float("-inf"))
    attn = torch.nan_to_num(scores.softmax(dim=-1))
    return attn @ v, attn

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); assert dim % heads == 0
        self.h, self.dk = heads, dim // heads
        self.qkv = nn.Linear(dim, 3*dim); self.proj = nn.Linear(dim, dim)
    def forward(self, x, key_mask, attn_bias=None):
        B, N, _ = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        out, attn = scaled_dot_product_attention(qkv[0], qkv[1], qkv[2],
            key_mask=key_mask[:, None, :] if key_mask is not None else None, attn_bias=attn_bias)
        return self.proj(out.transpose(1, 2).reshape(B, N, -1)), attn

class EncoderBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4, drop=0.1):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_ratio*dim), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(mlp_ratio*dim, dim))
        self.drop = nn.Dropout(drop)
    def forward(self, x, key_mask, attn_bias=None):
        a, attn = self.attn(self.ln1(x), key_mask, attn_bias)
        x = x + self.drop(a); x = x + self.mlp(self.ln2(x))
        return x, attn

class AttentionPooling(nn.Module):
    """Set Transformer PMA: one learnable class query attends over particles -> one jet vector."""
    def __init__(self, dim, heads):
        super().__init__(); self.h, self.dk = heads, dim // heads
        self.cls = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.q, self.kv, self.proj = nn.Linear(dim, dim), nn.Linear(dim, 2*dim), nn.Linear(dim, dim)
        self.last_attn = None
    def forward(self, x, key_mask):
        B, N, _ = x.shape
        q = self.q(self.cls.expand(B, 1, -1)).reshape(B, 1, self.h, self.dk).transpose(1, 2)
        kv = self.kv(x).reshape(B, N, 2, self.h, self.dk).permute(2, 0, 3, 1, 4)
        out, self.last_attn = scaled_dot_product_attention(q, kv[0], kv[1], key_mask=key_mask[:, None, :])
        return self.proj(out.transpose(1, 2).reshape(B, 1, -1)).squeeze(1)

In [ ]:
# --- ParT pairwise bias + CONFIGURABLE Particle Transformer ------------------------------
PAIR_NAMES = [r"$\ln\Delta$", r"$\ln k_T$", r"$\ln z$", r"$\ln m^2$"]

def pairwise_features(p4, eps=1e-8):
    """p4:(B,N,4)=(px,py,pz,E) -> U:(B,N,N,4)=[lnDelta, ln kT, ln z, ln m^2]."""
    px, py, pz, E = p4[..., 0], p4[..., 1], p4[..., 2], p4[..., 3]
    pt  = torch.sqrt(px**2 + py**2).clamp(min=1e-6); phi = torch.atan2(py, px)
    y   = 0.5 * torch.log(((E + pz).clamp(min=eps)) / ((E - pz).clamp(min=eps)))
    dy   = y[:, :, None] - y[:, None, :]
    dphi = phi[:, :, None] - phi[:, None, :]; dphi = torch.atan2(torch.sin(dphi), torch.cos(dphi))
    delta = torch.sqrt(dy**2 + dphi**2).clamp(min=1e-6)
    ptmin = torch.minimum(pt[:, :, None], pt[:, None, :])
    kt = (ptmin * delta).clamp(min=1e-6)
    z  = (ptmin / (pt[:, :, None] + pt[:, None, :] + eps)).clamp(min=1e-6, max=1.0)
    sE, spx = E[:, :, None]+E[:, None, :], px[:, :, None]+px[:, None, :]
    spy, spz = py[:, :, None]+py[:, None, :], pz[:, :, None]+pz[:, None, :]
    m2 = (sE**2 - spx**2 - spy**2 - spz**2).clamp(min=1e-6)
    return torch.stack([torch.log(delta), torch.log(kt), torch.log(z), torch.log(m2)], dim=-1)

class ParticleTransformer(nn.Module):
    """Configurable: use_pair_bias, pair_feat_idx (subset of the 4 features), pool in {'attn','mean'}."""
    def __init__(self, in_feat, n_classes=2, dim=64, depth=3, heads=4, drop=0.1,
                 use_pair_bias=True, pair_feat_idx=(0, 1, 2, 3), pool="attn"):
        super().__init__()
        self.use_pair_bias, self.pair_feat_idx, self.pool_kind = use_pair_bias, list(pair_feat_idx), pool
        self.embed = nn.Sequential(nn.Linear(in_feat, dim), nn.GELU(), nn.Linear(dim, dim))
        self.blocks = nn.ModuleList([EncoderBlock(dim, heads, drop=drop) for _ in range(depth)])
        self.pool = AttentionPooling(dim, heads) if pool == "attn" else None
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, n_classes))
        if use_pair_bias:
            self.pair_mlp = nn.Sequential(nn.Linear(len(self.pair_feat_idx), 16), nn.GELU(), nn.Linear(16, heads))
        self.last_attn, self.attn_maps = None, []
    def forward(self, x, mask, p4=None):
        h = self.embed(x); bias = None
        if self.use_pair_bias and p4 is not None:
            U = pairwise_features(p4)[..., self.pair_feat_idx]
            bias = self.pair_mlp(U).permute(0, 3, 1, 2)
        self.attn_maps = []
        for blk in self.blocks:
            h, self.last_attn = blk(h, mask, bias); self.attn_maps.append(self.last_attn)
        if self.pool_kind == "attn":
            z = self.pool(h, mask)
        else:
            z = (h * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1.0)   # masked mean
        return self.head(z)

def n_params(m): return sum(p.numel() for p in m.parameters())

In [ ]:
# --- load JetClass as masked sets: x (features), p4 (raw), pos (raw deta,dphi) -----------
import uproot, awkward as ak
CANDIDATE_PATHS = [
    "../data/JetClass_example_100k.root",
    "/Users/sanmay/Documents/ML_SCHOOLS/MLSCHOOL_2023_ICTS/Main_School/JetDataset/JetClass_example_100k.root",
    "./JetClass_example_100k.root",
]
root_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if root_path is None: raise FileNotFoundError("JetClass_example_100k.root not found")
print("Using:", root_path)

MAX_PART  = 128
N_PER_CLS = 2000     # per class; reduced from the module's 8000 so ~20 trainings stay fast
DIM, DEPTH, HEADS, EPOCHS = 64, 3, 4, 10

tree = uproot.open(root_path)["tree"]
br = tree.arrays(["part_px","part_py","part_pz","part_energy","part_deta","part_dphi",
                  "label_QCD","label_Tbqq","label_Tbl"])
is_qcd = ak.to_numpy(br["label_QCD"]).astype(bool)
is_top = ak.to_numpy(br["label_Tbqq"]).astype(bool) | ak.to_numpy(br["label_Tbl"]).astype(bool)
selidx = np.concatenate([np.where(is_qcd)[0][:N_PER_CLS], np.where(is_top)[0][:N_PER_CLS]])
labels = np.concatenate([np.zeros(N_PER_CLS), np.ones(N_PER_CLS)]).astype(np.int64)

px,py,pz,e = br["part_px"][selidx], br["part_py"][selidx], br["part_pz"][selidx], br["part_energy"][selidx]
deta,dphi  = br["part_deta"][selidx], br["part_dphi"][selidx]
pt = np.sqrt(px**2+py**2); dR = np.sqrt(deta**2+dphi**2)
sumpt, sume = ak.sum(pt,axis=1), ak.sum(e,axis=1)
FEATURE_NAMES = ["deta","dphi","log_pt","log_e","log_pt_rel","log_e_rel","dR"]
feat_list = [deta, dphi, np.log(pt+1e-8), np.log(e+1e-8),
             np.log(pt/sumpt+1e-8), np.log(e/sume+1e-8), dR]
def pad(f): return ak.to_numpy(ak.fill_none(ak.pad_none(f, MAX_PART, clip=True), 0.0))
X   = np.stack([pad(f) for f in feat_list], axis=-1).astype(np.float32)
P4  = np.stack([pad(px), pad(py), pad(pz), pad(e)], axis=-1).astype(np.float32)
POS = np.stack([pad(deta), pad(dphi)], axis=-1).astype(np.float32)                 # raw (eta,phi) for §5,§6
nparts = np.minimum(ak.to_numpy(ak.num(pt, axis=1)), MAX_PART)
MASK = (np.arange(MAX_PART)[None,:] < nparts[:,None]).astype(np.float32)

realmask = MASK.astype(bool); mean, std = X[realmask].mean(0), X[realmask].std(0)+1e-6
X = (X - mean) / std
Xt, P4t, POSt, Mt, Yt = map(torch.from_numpy, (X, P4, POS, MASK, labels))
g = torch.Generator().manual_seed(0); pm = torch.randperm(len(Yt), generator=g)
Xt, P4t, POSt, Mt, Yt = Xt[pm], P4t[pm], POSt[pm], Mt[pm], Yt[pm]
n = len(Yt); ntr, nva = int(0.70*n), int(0.15*n)
sl = {"train": slice(0,ntr), "val": slice(ntr,ntr+nva), "test": slice(ntr+nva,n)}

class JetSet(Dataset):
    def __init__(self, s): self.x, self.p4, self.m, self.y = Xt[s], P4t[s], Mt[s], Yt[s]
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.p4[i], self.m[i], self.y[i]
loaders = {k: DataLoader(JetSet(s), batch_size=64, shuffle=(k=="train")) for k, s in sl.items()}
print("jets:", {k: len(JetSet(s)) for k, s in sl.items()}, "| X", tuple(Xt.shape))

In [ ]:
# --- train / eval helpers (train_model stashes a per-epoch .history) ---------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps, ls, nt = [], [], 0.0, 0
    for x, p4, m, y in loader:
        x, p4, m, y = x.to(device), p4.to(device), m.to(device), y.to(device)
        logits = model(x, m, p4)
        ls += F.cross_entropy(logits, y, reduction="sum").item(); nt += y.size(0)
        ys.append(y.cpu()); ps.append(F.softmax(logits, -1)[:, 1].cpu())
    yv, pv = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return {"loss": ls/nt, "acc": ((pv>0.5).astype(int)==yv).mean(), "auc": roc_auc_score(yv, pv), "y": yv, "p": pv}

def background_rejection(y, p, eff=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    eps_b = max(np.interp(eff, tpr, fpr), 1.0/max(int((y==0).sum()), 1))
    return 1.0/eps_b

def train_model(model, loaders, epochs=EPOCHS, lr=1e-3, wd=1e-4, quiet=True):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = {"train_loss": [], "val_loss": [], "val_auc": []}
    t0 = time.time()
    for ep in range(epochs):
        model.train(); run, nb = 0.0, 0
        for x, p4, m, y in loaders["train"]:
            x, p4, m, y = x.to(device), p4.to(device), m.to(device), y.to(device)
            loss = F.cross_entropy(model(x, m, p4), y)
            opt.zero_grad(); loss.backward(); opt.step(); run += loss.item(); nb += 1
        sched.step(); va = evaluate(model, loaders["val"])
        hist["train_loss"].append(run/max(nb,1)); hist["val_loss"].append(va["loss"]); hist["val_auc"].append(va["auc"])
        if not quiet: print(f"  epoch {ep+1:2d}: train {run/max(nb,1):.3f}  val AUC {va['auc']:.4f}")
    model.history = hist; model.train_secs = time.time()-t0
    return model

In [ ]:
# --- base models reused across exercises: a plain Transformer and a full ParT ------------
print("Base Plain Transformer (no pairwise bias):")
base_plain = train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=HEADS,
                                             use_pair_bias=False), loaders, quiet=False)
print("Base Particle Transformer (all 4 pairwise features):")
base_part = train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=HEADS,
                                            use_pair_bias=True), loaders, quiet=False)
res_plain, res_part = evaluate(base_plain, loaders["test"]), evaluate(base_part, loaders["test"])
print(f"\nBASELINES  Plain AUC {res_plain['auc']:.4f}   ParT AUC {res_part['auc']:.4f}")
print("Setup complete.")

## Exercise 1 — Ablate the bias features

> *Retrain ParT with the pairwise bias restricted to only ln Δ, or only ln m². Which of the four features
> carries most of ParT's advantage?*

**The concept.** ParT's edge over a plain Transformer is the pairwise bias `U_ij` added to the attention
logits, built from four QCD-splitting kinematics: `lnΔ` (angular separation), `ln k_T` (splitting hardness),
`ln z` (momentum sharing), `ln m²` (pair invariant mass). To measure each feature's *value* (§5a only
*visualized* them), we retrain ParT with the bias restricted to a **single** feature at a time and compare
test AUC to the full-4 ParT and the no-bias plain Transformer. The gap `AUC(feature) − AUC(plain)` is that
feature's standalone contribution.

In [ ]:
abl = {"plain (none)": res_plain["auc"], "all 4": res_part["auc"]}
raw_names = ["lnΔ", "ln kT", "ln z", "ln m²"]
for j, nm in enumerate(raw_names):
    torch.manual_seed(0)
    m = train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=HEADS,
                                        use_pair_bias=True, pair_feat_idx=(j,)), loaders)
    abl[f"only {nm}"] = evaluate(m, loaders["test"])["auc"]
    print(f"only {nm:6s}: AUC {abl[f'only {nm}']:.4f}")
print(f"\nplain {abl['plain (none)']:.4f}   all-4 {abl['all 4']:.4f}")

order = ["plain (none)", "only lnΔ", "only ln kT", "only ln z", "only ln m²", "all 4"]
plt.figure(figsize=(7, 3.4)); plt.bar(order, [abl[k] for k in order])
plt.ylabel("test AUC"); plt.ylim(min(abl.values())-0.01, max(abl.values())+0.005)
plt.axhline(res_plain["auc"], ls="--", color="grey", label="plain (no bias)")
plt.title("Value of each ParT pairwise feature"); plt.xticks(rotation=25, ha="right"); plt.legend()
plt.tight_layout(); plt.show()

**What this shows.** Every single-feature bias already beats the plain Transformer — even one physics scalar
on the attention logits helps. The largest single jump usually comes from **`lnΔ`** (pure angular locality,
the same thing Module 2's k-NN hard-wired) and **`ln k_T`**/**`ln m²`** (which encode the *hardness* and
*mass* of the putative splitting — directly relevant to distinguishing a real 3-prong top decay from QCD).
`ln z` alone is weakest (momentum sharing is the least discriminating in isolation). The full-4 ParT is best,
but the ranking tells you the bias's power is mostly **angular + hardness** information — a compact, physically
interpretable prior. (Exact ordering shifts a little with scale/seed; the robust statement is *any* pairwise
feature > none, and angular/hardness features lead.)

## Exercise 2 — Depth / heads / width

> *Sweep `depth`, `heads`, `dim`. Where are the diminishing returns for jets, and how does the training curve
> change?*

**The concept.** Three capacity knobs: **depth** (how many rounds of all-to-all message passing), **width**
`dim` (representation size), and **heads** (how many relationship types attended in parallel). More capacity
helps until the ~100-particle jet task saturates, after which you pay compute/overfitting for nothing. We
sweep each knob around the base config (depth 3, dim 64, heads 4), holding the others fixed, and read off AUC
vs parameters.

In [ ]:
sweep = {}
def add(tag, model): sweep[tag] = dict(auc=evaluate(model, loaders["test"])["auc"], p=n_params(model))
add(f"base (d{DEPTH},w{DIM},h{HEADS})", base_part)
for depth in (1, 2, 5):
    torch.manual_seed(0)
    add(f"depth={depth}", train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=depth, heads=HEADS), loaders))
for dim in (32, 128):
    torch.manual_seed(0)
    add(f"dim={dim}", train_model(ParticleTransformer(len(FEATURE_NAMES), dim=dim, depth=DEPTH, heads=HEADS), loaders))
for heads in (1, 8):
    torch.manual_seed(0)
    add(f"heads={heads}", train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=heads), loaders))
for t, r in sweep.items(): print(f"{t:22s}: AUC {r['auc']:.4f}  ({r['p']:>6} params)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, (knob, keys) in zip(ax, [("depth", ["depth=1","depth=2",f"base (d{DEPTH},w{DIM},h{HEADS})","depth=5"]),
                                ("dim",   ["dim=32",f"base (d{DEPTH},w{DIM},h{HEADS})","dim=128"]),
                                ("heads", ["heads=1",f"base (d{DEPTH},w{DIM},h{HEADS})","heads=8"])]):
    xs = [int(k.split("=")[1]) if "=" in k else {"depth":DEPTH,"dim":DIM,"heads":HEADS}[knob] for k in keys]
    a.plot(xs, [sweep[k]["auc"] for k in keys], "o-"); a.set_xlabel(knob); a.set_ylabel("test AUC")
    a.set_title(f"AUC vs {knob}"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
# training curve of the base ParT
plt.figure(figsize=(5,3.6))
plt.plot(range(1, EPOCHS+1), base_part.history["val_auc"], "o-", label=f"base depth={DEPTH}")
plt.xlabel("epoch"); plt.ylabel("val AUC"); plt.title("Base ParT learning curve"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

**What this shows.** All three curves **rise then flatten**. Depth 1→2→3 buys real AUC; beyond ~3 blocks the
gain flattens (a ~100-particle jet does not need many rounds of global mixing). Width behaves similarly —
`dim=32` underfits, `64` is near the knee, `128` adds parameters for little. Heads help up to ~4 (several
relationship types) then saturate. The practical read: ParT for jets lives in a **small** regime (depth ≈ 3,
dim ≈ 64–128, heads ≈ 4–8); the physics bias does much of the heavy lifting, so raw capacity saturates early.
(At this reduced `N_PER_CLS` the absolute AUCs are below the module's ≈0.99; the *shape* of the diminishing
returns is the point and is scale-robust.)

## Exercise 3 — Class attention vs mean pool

> *Replace `AttentionPooling` with the masked mean-pool from Module 1. How much does learnable pooling
> actually help?*

**The concept.** The readout turns the set of particle tokens into one jet vector. Module 1 used a **masked
mean** (every particle equal); ParT uses **class attention** (PMA) — a learnable query that weights particles
by relevance. Our `ParticleTransformer(pool=...)` flips exactly this, keeping the encoder identical, so the
comparison isolates the pooling. We test it **with** the pairwise bias (full ParT) to see if learnable pooling
still helps once the encoder is already strong.

In [ ]:
torch.manual_seed(0)
part_mean = train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=HEADS,
                                            use_pair_bias=True, pool="mean"), loaders)
res_mean = evaluate(part_mean, loaders["test"])
print(f"ParT + class-attention pool (PMA): AUC {res_part['auc']:.4f}")
print(f"ParT + masked mean pool          : AUC {res_mean['auc']:.4f}")
print(f"Δ AUC from learnable pooling      : {res_part['auc']-res_mean['auc']:+.4f}")

**What this shows.** Class attention typically edges out mean pooling, but the gap is **modest** — usually a
few thousandths of AUC. The reason: after three self-attention blocks every token is already a context-aware
summary of the jet, so even a plain average is informative; the learnable query mainly sharpens the readout by
down-weighting soft junk. The lesson mirrors Module 1's attention-pooling exercise: learnable pooling helps
*most* when the upstream features are weak (raw DeepSets), and *least* when the encoder has already done the
mixing. It is a cheap, safe upgrade — just not where ParT's main advantage comes from (that's the bias, §1).

## Exercise 4 — ISAB accuracy trade-off

> *§10a benchmarked ISAB's speed. Now swap the encoder's full-attention blocks for `ISAB` (m=16, 32), retrain,
> and measure the AUC you trade for the linear scaling.*

**The concept.** Full self-attention (SAB) is `O(N²)`; the Set Transformer's **ISAB** routes all-to-all
attention through `m ≪ N` learnable *inducing points* (`X→I→X`), costing `O(Nm)` — the win for
high-multiplicity events. The question is what accuracy you pay. We build a Set-Transformer classifier whose
encoder is either full **SAB** or **ISAB(m)**, keep the PMA readout, and compare AUC at `m = 16, 32`. (ISAB
attends through inducing points, so it does not carry ParT's N×N pairwise bias — this is a pure
architecture-vs-cost comparison, run without the bias.)

In [ ]:
class MAB(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=2):
        super().__init__(); self.h, self.dk = heads, dim//heads
        self.lnA, self.lnB, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.q, self.kv, self.proj = nn.Linear(dim, dim), nn.Linear(dim, 2*dim), nn.Linear(dim, dim)
        self.ff = nn.Sequential(nn.Linear(dim, mlp_ratio*dim), nn.GELU(), nn.Linear(mlp_ratio*dim, dim))
    def forward(self, A, B, key_mask=None):
        Aw, Bw = self.lnA(A), self.lnB(B)
        q = self.q(Aw).reshape(*Aw.shape[:2], self.h, self.dk).transpose(1, 2)
        kv = self.kv(Bw).reshape(*Bw.shape[:2], 2, self.h, self.dk).permute(2, 0, 3, 1, 4)
        out, _ = scaled_dot_product_attention(q, kv[0], kv[1],
                    key_mask=key_mask[:, None, :] if key_mask is not None else None)
        x = A + self.proj(out.transpose(1, 2).reshape(*Aw.shape[:2], -1))
        return x + self.ff(self.ln2(x))

class SAB(nn.Module):
    def __init__(self, dim, heads): super().__init__(); self.mab = MAB(dim, heads)
    def forward(self, X, key_mask=None): return self.mab(X, X, key_mask)

class ISAB(nn.Module):
    def __init__(self, dim, heads, m=16):
        super().__init__(); self.I = nn.Parameter(torch.randn(1, m, dim)*0.02)
        self.mab1, self.mab2 = MAB(dim, heads), MAB(dim, heads)
    def forward(self, X, key_mask=None):
        H = self.mab1(self.I.expand(X.shape[0], -1, -1), X, key_mask)   # inducing points can't be masked out
        return self.mab2(X, H)

class SetTransformerClf(nn.Module):
    """Encoder of SAB or ISAB blocks + PMA readout (no pairwise bias)."""
    def __init__(self, in_feat, dim=DIM, depth=DEPTH, heads=HEADS, block="sab", m=16):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(in_feat, dim), nn.GELU(), nn.Linear(dim, dim))
        mk = (lambda: SAB(dim, heads)) if block == "sab" else (lambda: ISAB(dim, heads, m))
        self.blocks = nn.ModuleList([mk() for _ in range(depth)])
        self.pool = AttentionPooling(dim, heads); self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 2))
    def forward(self, x, mask, p4=None):
        h = self.embed(x)
        for blk in self.blocks: h = blk(h, mask)
        return self.head(self.pool(h, mask))

isab_res = {}
torch.manual_seed(0); isab_res["SAB (full O(N²))"] = evaluate(train_model(SetTransformerClf(len(FEATURE_NAMES), block="sab"), loaders), loaders["test"])["auc"]
for mm in (16, 32):
    torch.manual_seed(0)
    isab_res[f"ISAB m={mm} (O(Nm))"] = evaluate(train_model(SetTransformerClf(len(FEATURE_NAMES), block="isab", m=mm), loaders), loaders["test"])["auc"]
for k, v in isab_res.items(): print(f"{k:22s}: AUC {v:.4f}")

**What this shows.** ISAB recovers **most** of full attention's AUC — with `m=32` inducing points it is
usually within a few thousandths of SAB, and `m=16` a touch behind. You trade a small, tunable accuracy hit
for attention that scales **linearly** in the number of particles (the speed-up §10a benchmarked). For our
~50–100-particle jets full attention is cheap, so SAB wins on accuracy; ISAB earns its keep when `N` is large
— pileup, full-event point clouds, tracking hits — where `O(N²)` is prohibitive. `m` is the knob that dials
between the two regimes.

## Exercise 5 — Attention → graph: does it rediscover the k-NN?

> *Threshold a trained attention map at its top-k entries per row and compare the induced graph, jet-by-jet,
> to ParticleNet's k-NN graph. Does attention rediscover the same neighbours?*

**The concept.** §9a showed attention *enhances* small-ΔR pairs on average. Here we make it concrete and
per-jet: for each particle we take its **top-k most-attended** partners (from the base ParT's last block) and
compare that set to its **k nearest neighbours in (η, φ)** — exactly the graph ParticleNet's first block
builds. We report the mean overlap fraction against the random baseline `k/(n−1)`. High overlap = attention
learned, from data, the angular locality Module 2 imposed by hand.

In [ ]:
@torch.no_grad()
def attn_knn_overlap(model, k=8, nbatch=8):
    ovs, bases = [], []
    for bi, (x, p4, m, y) in enumerate(loaders["test"]):
        if bi >= nbatch: break
        x, p4, m = x.to(device), p4.to(device), m.to(device)
        model(x, m, p4)
        A = model.last_attn.mean(1)                                     # (B,N,N) last block, head-averaged
        B, N, _ = A.shape
        # physical (eta,phi) from p4 -> kNN neighbours (translation-invariant, so absolute coords are fine)
        px, py, pz, E = p4.unbind(-1); ptt = (px**2+py**2).sqrt().clamp(min=1e-6)
        eta = torch.asinh(pz/ptt); phi = torch.atan2(py, px)
        pos = torch.stack([eta, phi], -1)
        d2 = (pos[:, :, None, :]-pos[:, None, :, :]).pow(2).sum(-1)
        big = 1e9; keymask = m[:, None, :].bool(); eye = torch.eye(N, device=x.device).bool()[None]
        d2 = d2.masked_fill(~keymask, big).masked_fill(eye, big)
        A = A.masked_fill(~keymask, -big).masked_fill(eye, -big)
        knn = d2.topk(k, dim=-1, largest=False).indices                # (B,N,k)
        att = A.topk(k, dim=-1, largest=True).indices
        oh_knn = torch.zeros_like(d2).scatter_(-1, knn, 1.0)
        oh_att = torch.zeros_like(d2).scatter_(-1, att, 1.0)
        inter = (oh_knn*oh_att).sum(-1)                                 # (B,N) overlap count / query
        real = m.bool()
        nreal = m.sum(1, keepdim=True).clamp(min=2)
        ovs.append((inter/k)[real].cpu().numpy())
        bases.append((k/(nreal-1)).expand(-1, N)[real].cpu().numpy())
    return np.concatenate(ovs).mean(), np.concatenate(bases).mean()

for lbl, mdl in [("Base ParT", base_part), ("Base Plain", base_plain)]:
    ov, base = attn_knn_overlap(mdl, k=8)
    print(f"{lbl:11s}: mean top-8 attention∩kNN overlap = {ov:.2f}   (random baseline {base:.2f})")

**What this shows.** The trained attention's top-k partners overlap with the (η, φ) k-NN **well above the
random baseline** — attention *rediscovers* angular locality without ever being told to, confirming §9a at the
level of individual neighbour sets. ParT overlaps more than the plain Transformer because its `lnΔ` bias
explicitly rewards small separations. But the overlap is **not 1.0**: attention also builds *non-local* edges
(hard prong ↔ hard prong across the jet) that a fixed k-NN cannot, which is exactly the extra expressivity a
soft, learned graph buys over Module 2's hard one.

## Exercise 6 — Three-way race: PFN vs ParticleNet vs ParT

> *Put PFN (M1), ParticleNet (M2) and ParT (M3) on one ROC / rejection plot for the same test jets.*

**The concept.** The course's three archetypes on one axis: **no edges** (PFN / DeepSets), **hard k-NN edges**
(ParticleNet), **soft all-to-all edges** (ParT). We train a PFN (masked-sum DeepSets on the padded features)
and a ParticleNet (Module 2's PyG implementation, built from the **same** jet indices) and overlay their ROCs
with the base ParT — all evaluated on the identical test jets. Expect a clean ordering
**PFN < ParticleNet ≤ ParT**, with the biggest separation at high background rejection.

In [ ]:
# PFN over the padded set (masked-sum DeepSets), same loaders as ParT
class PFN(nn.Module):
    def __init__(self, in_feat, hidden=(64,128,128), latent=128):
        super().__init__()
        dims=[in_feat,*hidden]; layers=[]
        for a,b in zip(dims[:-1],dims[1:]): layers += [nn.Linear(a,b), nn.ReLU()]
        self.phi = nn.Sequential(*layers, nn.Linear(hidden[-1], latent))
        self.head = nn.Sequential(nn.Linear(latent,128), nn.ReLU(), nn.Linear(128,2))
    def forward(self, x, mask, p4=None):
        h = self.phi(x); z = (h*mask.unsqueeze(-1)).sum(1)
        return self.head(z)
torch.manual_seed(0); pfn = train_model(PFN(len(FEATURE_NAMES)), loaders); res_pfn = evaluate(pfn, loaders["test"])
print(f"PFN AUC {res_pfn['auc']:.4f}")

In [ ]:
# ParticleNet via PyG, built from the SAME shuffled indices -> identical test jets
import torch_geometric
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader

def knn_graph_native(x, k, batch):
    with torch.no_grad():
        N=x.size(0); x2=(x*x).sum(1,keepdim=True); d2=x2+x2.t()-2*(x@x.t())
        same=batch.unsqueeze(0)==batch.unsqueeze(1); d2=d2.masked_fill(~same,float("inf")); d2.fill_diagonal_(float("inf"))
        kk=min(k,max(N-1,1)); dk,idx=d2.topk(kk,dim=1,largest=False); valid=torch.isfinite(dk)
        c=torch.arange(N,device=x.device).unsqueeze(1).expand(-1,kk)
        return torch.stack([idx[valid], c[valid]], 0)
class EdgeConv(MessagePassing):
    def __init__(self, ic, oc, aggr="mean"):
        super().__init__(aggr=aggr)
        self.mlp=nn.Sequential(nn.Linear(2*ic,oc),nn.BatchNorm1d(oc),nn.ReLU(),nn.Linear(oc,oc),nn.BatchNorm1d(oc),nn.ReLU())
    def forward(self,x,ei): return self.propagate(ei,x=x)
    def message(self,x_i,x_j): return self.mlp(torch.cat([x_i,x_j-x_i],-1))
class Block(nn.Module):
    def __init__(self,ic,oc): super().__init__(); self.c=EdgeConv(ic,oc); self.s=nn.Linear(ic,oc)
    def forward(self,x,ei): return F.relu(self.c(x,ei)+self.s(x))
class ParticleNet(nn.Module):
    def __init__(self, inf, k=8, ch=(32,64,64), fc=128):
        super().__init__(); self.k=k; chs=[inf,*ch]
        self.blocks=nn.ModuleList([Block(chs[i],chs[i+1]) for i in range(len(ch))])
        self.head=nn.Sequential(nn.Linear(ch[-1],fc),nn.ReLU(),nn.Dropout(0.1),nn.Linear(fc,2))
    def forward(self,data):
        batch=data.batch; feats,coords=data.x,data.pos
        for blk in self.blocks:
            ei=knn_graph_native(coords,self.k,batch); feats=blk(feats,ei); coords=feats
        return self.head(global_mean_pool(feats,batch))

def make_geo(split):
    s=sl[split]; xs,ps,ms,ys=Xt[s],POSt[s],Mt[s],Yt[s]; dl=[]
    for i in range(len(ys)):
        nn_=int(ms[i].sum())
        if nn_<2: continue
        dl.append(Data(x=xs[i,:nn_], pos=ps[i,:nn_], y=ys[i].view(1)))
    return dl
geo = {k: GeoLoader(make_geo(k), batch_size=64, shuffle=(k=="train")) for k in ("train","val","test")}

def train_pnet(model, epochs=EPOCHS):
    model=model.to(device); opt=torch.optim.Adam(model.parameters(),1e-3)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    for ep in range(epochs):
        model.train()
        for d in geo["train"]:
            d=d.to(device); loss=F.cross_entropy(model(d),d.y); opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model
@torch.no_grad()
def eval_pnet(model):
    model.eval(); ys,ps=[],[]
    for d in geo["test"]:
        d=d.to(device); ps.append(F.softmax(model(d),-1)[:,1].cpu()); ys.append(d.y.cpu())
    y,p=torch.cat(ys).numpy(),torch.cat(ps).numpy()
    return {"y":y,"p":p,"auc":roc_auc_score(y,p)}
torch.manual_seed(0); pnet=train_pnet(ParticleNet(len(FEATURE_NAMES),k=8)); res_pnet=eval_pnet(pnet)
print(f"ParticleNet AUC {res_pnet['auc']:.4f}")

In [ ]:
print(f"{'εS':>5} {'PFN':>9} {'PNet':>9} {'ParT':>9}")
for eff in (0.3, 0.5, 0.7):
    print(f"{eff:5.1f} "
          f"{background_rejection(res_pfn['y'],res_pfn['p'],eff):9.1f} "
          f"{background_rejection(res_pnet['y'],res_pnet['p'],eff):9.1f} "
          f"{background_rejection(res_part['y'],res_part['p'],eff):9.1f}")
plt.figure(figsize=(5.4,4.3))
for r, lbl in [(res_pfn,"PFN (M1, no edges)"), (res_pnet,"ParticleNet (M2, k-NN)"), (res_part,"ParT (M3, soft)")]:
    fpr,tpr,_=roc_curve(r["y"],r["p"]); plt.plot(tpr,1.0/np.clip(fpr,1e-6,1),label=f"{lbl}  AUC {r['auc']:.3f}")
plt.yscale("log"); plt.xlabel("signal efficiency $\\epsilon_S$ (top)"); plt.ylabel("background rejection $1/\\epsilon_B$")
plt.grid(True,which="both",alpha=0.3); plt.legend(); plt.title("Three archetypes, same test jets"); plt.show()

**What this shows.** The ordering **PFN < ParticleNet ≤ ParT** comes out as expected, and the gaps widen at
the **high-rejection** end — the demanding working points analyses care about. Each step corresponds to one
new ingredient in the course's story: PFN pools independent particles; ParticleNet adds *hard* k-NN edges;
ParT makes those edges *soft, all-to-all, and physics-biased*. At this reduced training size the absolute
numbers trail the modules' headline AUCs (PFN≈0.97, PNet≈0.99, ParT≈0.99), but the **relative** staircase is
the robust, teachable result. (ParT and ParticleNet are often neck-and-neck at small scale; ParT pulls ahead
with more data — the inductive-bias↔data theme again.)

## Exercise 7 — Boost augmentation

> *The bias is longitudinal-boost invariant (§5a) but the input features (log_pt, log_e, …) are not. Add random
> longitudinal boosts as data augmentation: does it help, and does ParT — already partly boost-aware through
> its bias — benefit less than the plain Transformer?*

**The concept.** A longitudinal (beam-direction) boost by rapidity ξ leaves `pT` and φ invariant but shifts
rapidity by ξ and changes `E` (`E' = E cosh ξ + pz sinh ξ`). So the *input* features that use energy/pseudo-
rapidity (`log_e`, `log_e_rel`, `deta`, `dR`) **move under a boost**, while ParT's pairwise bias — built from
rapidity *differences*, `pT`, and `m²` — is **invariant**. We therefore expect: (i) a boosted test set hurts
models trained only at rest; (ii) boost **augmentation** recovers robustness; (iii) ParT, already partly
boost-aware, degrades **less** and needs augmentation **less** than the plain Transformer. We featurize
directly from `p4` (self-consistently) so boosting is exact.

In [ ]:
def featurize_p4(p4, mask, stats=None):
    """(B,N,4) raw 4-vectors -> (B,N,7) standardized features, matching FEATURE_NAMES, computed FROM p4."""
    px, py, pz, E = p4.unbind(-1)
    pt = (px**2+py**2).sqrt().clamp(min=1e-6); phi = torch.atan2(py, px)
    eta = torch.asinh(pz/pt)
    m = mask.bool()
    spx=(px*mask).sum(1,keepdim=True); spy=(py*mask).sum(1,keepdim=True)
    spz=(pz*mask).sum(1,keepdim=True); sE=(E*mask).sum(1,keepdim=True)
    ptj=(spx**2+spy**2).sqrt().clamp(min=1e-6)
    etaj=torch.asinh(spz/ptj); phij=torch.atan2(spy,spx)
    deta=eta-etaj; dphi=torch.atan2(torch.sin(phi-phij), torch.cos(phi-phij))
    sumpt=(pt*mask).sum(1,keepdim=True).clamp(min=1e-6); sume=(E*mask).sum(1,keepdim=True).clamp(min=1e-6)
    feats=torch.stack([deta, dphi, torch.log(pt), torch.log(E.clamp(min=1e-6)),
                       torch.log(pt/sumpt), torch.log(E.clamp(min=1e-6)/sume),
                       (deta**2+dphi**2).sqrt()], -1)
    feats=feats*mask.unsqueeze(-1)
    if stats is None:
        fl=feats[m]; stats=(fl.mean(0), fl.std(0)+1e-6)
    return (feats-stats[0])/stats[1]*mask.unsqueeze(-1), stats

def boost_z(p4, xi):
    px,py,pz,E = p4.unbind(-1); ch,sh = math.cosh(xi), math.sinh(xi)
    return torch.stack([px, py, pz*ch+E*sh, E*ch+pz*sh], -1)

# training statistics from the un-boosted train set
_, STATS = featurize_p4(P4t[sl["train"]].to(device), Mt[sl["train"]].to(device))

def train_boost(use_bias, augment, xi_max=2.0, epochs=8):
    torch.manual_seed(0)
    model = ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=HEADS, use_pair_bias=use_bias).to(device)
    opt = torch.optim.AdamW(model.parameters(), 1e-3, weight_decay=1e-4)
    for ep in range(epochs):
        model.train()
        for x, p4, m, y in loaders["train"]:
            p4, m, y = p4.to(device), m.to(device), y.to(device)
            if augment:
                xi = float(np.random.uniform(-xi_max, xi_max)); p4b = boost_z(p4, xi)
            else:
                p4b = p4
            xb, _ = featurize_p4(p4b, m, STATS)
            loss = F.cross_entropy(model(xb, m, p4b), y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def auc_at_boost(model, xi):
    ys, ps = [], []
    for x, p4, m, y in loaders["test"]:
        p4, m = p4.to(device), m.to(device); p4b = boost_z(p4, xi)
        xb, _ = featurize_p4(p4b, m, STATS)
        ps.append(F.softmax(model(xb, m, p4b), -1)[:,1].cpu()); ys.append(y)
    return roc_auc_score(torch.cat(ys).numpy(), torch.cat(ps).numpy())

models = {("plain","no-aug"): train_boost(False, False), ("plain","aug"): train_boost(False, True),
          ("ParT","no-aug"):  train_boost(True,  False), ("ParT","aug"):  train_boost(True,  True)}
xis = [0.0, 0.5, 1.0, 2.0, 3.0]
print(f"{'model':16s} " + " ".join(f"ξ={x}" for x in xis))
curves = {}
for key, mdl in models.items():
    curves[key] = [auc_at_boost(mdl, x) for x in xis]
    print(f"{key[0]+'/'+key[1]:16s} " + " ".join(f"{a:.3f}" for a in curves[key]))

In [ ]:
plt.figure(figsize=(6,4))
for key, c in curves.items():
    ls = "-" if key[1]=="aug" else "--"; col = "C3" if key[0]=="ParT" else "C0"
    plt.plot(xis, c, ls+"o", color=col, label=f"{key[0]} ({key[1]})")
plt.xlabel("test-time longitudinal boost ξ"); plt.ylabel("test AUC")
plt.title("Robustness to boost: augmentation vs built-in invariance"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()
drop = {k: curves[k][0]-curves[k][-1] for k in curves}
print("AUC drop from ξ=0 to ξ=3 (smaller = more boost-robust):")
for k in curves: print(f"  {k[0]+'/'+k[1]:16s}: {drop[k]:+.3f}")

**What this shows.** Read the **slopes**, not the absolute heights. The **plain, no-aug** model falls off
fastest as the test boost grows — its energy/η-based inputs drift out of distribution. **Augmentation flattens
the plain model's curve dramatically** (it *learns* the invariance from boosted examples). **ParT degrades
less even without augmentation**, because its pairwise bias is boost-invariant by construction, so a large
part of its computation never sees the boost — and it therefore *benefits less* from augmentation (its
no-aug and aug curves are closer together). This is the module's headline trade-off made quantitative: you
can **bake in** a symmetry (ParT's bias, cheap and exact) or **learn** it from augmented data (works, but
costs data/epochs and is only approximate) — and Module 4 takes "bake it in" all the way to full equivariance.

## Exercise 8 — Faithful attribution: gradient-weighted rollout

> *§9e showed attention and gradient saliency only partly agree. Implement gradient-weighted attention rollout
> (Chefer et al., 2021): before composing, weight each layer's attention by
> E_h[(∂y_c/∂A_ℓ) ⊙ A_ℓ]₊. Does the importance map shift onto the particles the gradient says matter?*

**The concept.** Plain rollout (§9d) composes the raw per-layer attention `Ā_ℓ` — but it is *class-agnostic*
(it ignores which class you are explaining). **Chefer et al.** make it class-specific by weighting each layer
by its **gradient w.r.t. the target logit**: `Ā_ℓ ← (∇_{A_ℓ} y_c ⊙ A_ℓ)₊`, head-averaged, before composing
with the residual-identity and folding in the (grad-weighted) class-attention readout. The prediction is that
this map aligns **better with gradient saliency** than plain rollout, because it is built from the gradient.
We measure the top-k overlap with input-gradient saliency for both.

In [ ]:
def _norm(imp, m): return (imp*m) / (imp*m).sum(-1, keepdim=True).clamp(min=1e-9)

@torch.no_grad()
def class_importance(model, x, p4, m):
    model(x, m, p4); imp = model.pool.last_attn.mean(1).squeeze(1)
    return _norm(imp, m)

@torch.no_grad()
def rollout_importance(model, x, p4, m):
    model(x, m, p4); B, N = m.shape; I = torch.eye(N, device=x.device)[None]; R = I.repeat(B,1,1)
    for A in model.attn_maps:
        Ah = 0.5*A.mean(1) + 0.5*I; Ah = Ah/Ah.sum(-1,keepdim=True).clamp(min=1e-9); R = Ah @ R
    p = model.pool.last_attn.mean(1)
    return _norm((p @ R).squeeze(1), m)

def grad_rollout_importance(model, x, p4, m, cls=1):
    model.zero_grad(set_to_none=True)
    logit = model(x, m, p4)[:, cls].sum()
    tensors = list(model.attn_maps) + [model.pool.last_attn]
    grads = torch.autograd.grad(logit, tensors, retain_graph=False)   # dY/dA for every attention map
    with torch.no_grad():
        B, N = m.shape; I = torch.eye(N, device=x.device)[None]; R = I.repeat(B,1,1)
        for A, gA in zip(model.attn_maps, grads[:-1]):
            w = (gA * A.detach()).clamp(min=0).mean(1)               # E_h[(dY/dA ⊙ A)_+]  -> (B,N,N)
            Ah = w + I; Ah = Ah/Ah.sum(-1,keepdim=True).clamp(min=1e-9); R = Ah @ R
        pw = (grads[-1] * model.pool.last_attn.detach()).clamp(min=0).mean(1)   # grad-weighted class attn (B,1,N)
        return _norm((pw @ R).squeeze(1), m)

def grad_saliency(model, x, p4, m):
    x = x.clone().requires_grad_(True)
    gsal, = torch.autograd.grad(model(x, m, p4)[:,1].sum(), x)
    return (gsal.abs().sum(-1)*m).detach()

def overlap_at_k(a, b, m, k):
    big=(1-m)*-1e9; ta,tb=(a+big).topk(k,-1).indices,(b+big).topk(k,-1).indices
    return float(np.mean([len(set(ta[i].tolist())&set(tb[i].tolist()))/k for i in range(a.shape[0])]))

sel = (Yt==1).nonzero().squeeze(1); sel = sel[sel>=sl["test"].start][:256]
x, p4, m = Xt[sel].to(device), P4t[sel].to(device), Mt[sel].to(device)
sal = grad_saliency(base_part, x, p4, m)
imp_class = class_importance(base_part, x, p4, m)
imp_roll  = rollout_importance(base_part, x, p4, m)
imp_groll = grad_rollout_importance(base_part, x, p4, m)
print("Top-k overlap with input-gradient saliency (higher = more faithful):")
print(f"{'k':>3} {'class-attn':>11} {'rollout':>9} {'grad-rollout':>13}")
for k in (3, 5, 10):
    print(f"{k:>3} {overlap_at_k(imp_class,sal,m,k):>11.2f} {overlap_at_k(imp_roll,sal,m,k):>9.2f} {overlap_at_k(imp_groll,sal,m,k):>13.2f}")

In [ ]:
# Visualize all three attributions on one top jet
from math import asinh
one = sel[:1]; x1, p41, m1 = Xt[one].to(device), P4t[one].to(device), Mt[one].to(device)
n = int(m1[0].sum())
px,py,pz,E = P4t[one][0,:n].unbind(-1); pt=(px**2+py**2).sqrt(); eta=torch.asinh(pz/pt.clamp(min=1e-6)); phi=torch.atan2(py,px)
w=pt; y0=(w*eta).sum()/w.sum(); p0=torch.atan2((w*phi.sin()).sum(),(w*phi.cos()).sum())
dy=(eta-y0).numpy(); dp=torch.atan2((phi-p0).sin(),(phi-p0).cos()).numpy(); ptn=pt.numpy()
maps = [("class-attn (last layer)", class_importance(base_part,x1,p41,m1)[0,:n].detach().cpu().numpy()),
        ("plain rollout",           rollout_importance(base_part,x1,p41,m1)[0,:n].detach().cpu().numpy()),
        ("grad-weighted rollout",   grad_rollout_importance(base_part,x1,p41,m1)[0,:n].detach().cpu().numpy())]
fig, ax = plt.subplots(1, 3, figsize=(14,4.3), sharex=True, sharey=True)
for a,(t,ip) in zip(ax, maps):
    sc=a.scatter(dp, dy, s=15+300*ptn/ptn.max(), c=ip, cmap="plasma", edgecolor="k", linewidth=.3)
    a.set_title(t); a.set_xlabel(r"$\Delta\phi$"); fig.colorbar(sc,ax=a,fraction=.046)
ax[0].set_ylabel(r"$\Delta y$"); fig.suptitle("Three attributions of the same top jet (area ∝ pT)")
fig.tight_layout(); plt.show()

**What this shows.** Gradient-weighted rollout typically has the **highest overlap with input-gradient
saliency** of the three — unsurprising, since it is *built from* that gradient, but reassuring that it does so
while keeping rollout's layer-composition structure (it is class-specific where plain rollout is not). Plain
rollout and last-layer class-attention are class-agnostic lenses; they can spread importance onto particles
the classifier does not actually use. The practical guidance from the *"Attention is/► is not Explanation"*
debate holds: **corroborate attention with a gradient-aware method** before trusting an attention picture.
Chefer's method is the standard, cheap way to do exactly that. (Overlaps are modest in absolute terms — these
are *approximations* of a nonlinear model — but the **ranking** grad-rollout ≳ rollout ≈ class-attn is the
point.)

## Recap

| Exercise | Result |
|---|---|
| 1 · ablate bias | any pairwise feature > none; **angular (lnΔ) + hardness (ln kT / ln m²)** lead |
| 2 · depth/heads/width | AUC saturates early — jets live in a small-capacity regime |
| 3 · class-attn vs mean | learnable pooling helps a little; the encoder already did the mixing |
| 4 · ISAB | recovers most AUC at `O(Nm)`; the win is for large-N events |
| 5 · attention→graph | attention rediscovers the (η,φ) k-NN above chance, plus non-local edges |
| 6 · three-way race | **PFN < ParticleNet ≤ ParT**, gaps largest at high purity |
| 7 · boost aug | inputs drift under boost; ParT's invariant bias makes it more robust, less aug-dependent |
| 8 · faithful attribution | grad-weighted rollout aligns best with saliency — corroborate before trusting |

Attention is message passing on a **soft, learned, all-to-all** graph. The next step is the last symmetry we
have not yet baked in — **rotations and Lorentz boosts**. On to Module 4: equivariant networks.